# Compare several embedding methods
- text-embedding-ada-002
- BoW
- tf-idf
- BERT

In [ ]:
import pandas as pd
import numpy as np
import random
import torch

from sentence_transformers import SentenceTransformer

## Load all data

### Read in processed datasets and original FoodOn dataset for correct labels

In [ ]:
# Read in the processed datasets
df_nevo = pd.read_pickle("../data/intermediate/df_nevo_processed.pkl")
df_fooddatacentral = pd.read_pickle("../data/intermediate/df_fooddatacentral_processed.pkl")
df_foodon = pd.read_pickle("../data/intermediate/df_foodon_processed.pkl")
df_kap = pd.read_json("../data/input/df_kap.json")

In [ ]:
# Read in original FoodOn dataset and merge with processed FoodOn dataset to get the original label corresponding to the processed food name (to check the correctness)
foodon_path = '../data/input/FOODON_class_label.csv'
df_foodon_class_label = pd.read_csv(foodon_path)
df_foodon_combined = pd.merge(df_foodon, df_foodon_class_label, how="left", left_on="id", right_on="Class ID")
df_foodon_combined.head()

In [ ]:
# Get all names
df_nevo_names = df_nevo['name']
df_fooddatacentral_names = df_fooddatacentral['name']
df_foodon_names = df_foodon['name']
df_kap_names = df_kap['name']

### Load labels from datasets

In [ ]:
# Load existing NEVO labels
nevo_label_path = "../data/input/labels/nevo_labels.xlsx"
nevo_labels = pd.read_excel(nevo_label_path)

In [ ]:
# Load existing FoodData Central labels
fdc_label_path = "../data/input/labels/fdc_labels.xlsx"
fdc_labels = pd.read_excel(fdc_label_path)

In [ ]:
# Load existing KAP labels
kap_label_path = "../data/input/labels/kap_labels.xlsx"
kap_labels = pd.read_excel(kap_label_path)

### Method 1: text-embedding-ada-002

In [ ]:
from helpers.step2 import OpenAITextSimilarityProcessor, add_best_match_traditional, determine_results_traditional

**NEVO**

In [ ]:
## NEVO

# Get NEVO food names
nevo_names = list(set(nevo_labels['nevo_name']))

# Create processor instance
openai_nevo_processor = OpenAITextSimilarityProcessor(df1=df_nevo, df2=df_foodon, df1_embedding_column='embedding', df2_embedding_column='embedding')

# Compute similarity between nevo food items and FoodOn classes in matrix 
openai_nevo_processor.compute_cosine_similarity()

# Find top 15 FoodOn matches
openai_nevo_processor.find_top_k(names = nevo_names)

# Aggregate results
openai_nevo_processor.aggregate_results()

# Get results as a DataFrame
df_nevo_openai = openai_nevo_processor.get_results()
df_nevo_openai.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
df_nevo_openai.head(5)

In [ ]:
nevo_openai_merged_data = add_best_match_traditional(df_nevo_openai, nevo_labels, df_foodon_combined, 'nevo_name')
nevo_openai_merged_data.head()

In [ ]:
openai_nevo_results = determine_results_traditional(nevo_openai_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
openai_nevo_results

**FoodData Central**

In [ ]:
## FDC

# Get FDC food names
fdc_names = list(set(fdc_labels['orig_name']))

# Create processor instance
openai_fdc_processor = OpenAITextSimilarityProcessor(df1=df_fooddatacentral, df2=df_foodon, df1_embedding_column='embedding', df2_embedding_column='embedding')

# Compute similarity between kap food items and FoodOn classes in matrix 
openai_fdc_processor.compute_cosine_similarity()

# Find top FoodOn matches
openai_fdc_processor.find_top_k(names=fdc_names)

# Aggregate results
openai_fdc_processor.aggregate_results()

# Get results as a DataFrame
df_fdc_openai = openai_fdc_processor.get_results()
df_fdc_openai.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
df_fdc_openai.head(5)

In [ ]:
fdc_openai_merged_data = add_best_match_traditional(df_fdc_openai, fdc_labels, df_foodon_combined, 'orig_name')
fdc_openai_merged_data.head()

In [ ]:
openai_fdc_results = determine_results_traditional(fdc_openai_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
openai_fdc_results

**KAP**

In [ ]:
## KAP

# Get KAP food names
kap_names = list(set(kap_labels['orig_name']))

# Create processor instance
openai_kap_processor = OpenAITextSimilarityProcessor(df1=df_kap, df2=df_foodon, df1_embedding_column='embedding', df2_embedding_column='embedding')

# Compute similarity between kap food items and FoodOn classes in matrix 
openai_kap_processor.compute_cosine_similarity()

# Find top 15 FoodOn matches
openai_kap_processor.find_top_k(names=kap_names)

# Aggregate results
openai_kap_processor.aggregate_results()

# Get results as a DataFrame
df_kap_openai = openai_kap_processor.get_results()
df_kap_openai.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
df_kap_openai.head(5)

In [ ]:
kap_openai_merged_data = add_best_match_traditional(df_kap_openai, kap_labels, df_foodon_combined, 'orig_name')
kap_openai_merged_data.head()

In [ ]:
openai_kap_results = determine_results_traditional(kap_openai_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
openai_kap_results

### Method 2: BoW

**NEVO**

In [ ]:
from helpers.step2 import TextSimilarityProcessor

In [ ]:
# Create processor instance
bow_nevo_processor = TextSimilarityProcessor(df_nevo_names, df_foodon_names, vectorizer_type='bow')

# Compute shared dictionary
bow_nevo_processor.fit_vectorizer()

# Create BoW vectors specific for each dataset
bow_nevo_processor.transform_to_vec()

# Compute similarity between Nevo food items and FoodOn classes in matrix
bow_nevo_processor.compute_cosine_similarity()

nevo_names = list(set(nevo_labels['nevo_name']))
bow_nevo_processor.find_top_k(names=nevo_names)

# Aggregate results
bow_nevo_processor.aggregate_results()
df_nevo_bow = bow_nevo_processor.get_results()
df_nevo_bow.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
df_nevo_bow.head()

In [ ]:
# Nevo label matches
nevo_bow_merged_data = add_best_match_traditional(df_nevo_bow, nevo_labels, df_foodon_combined, 'nevo_name')
nevo_bow_merged_data.head()

In [ ]:
bow_nevo_results = determine_results_traditional(nevo_bow_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
bow_nevo_results

**Fooddatacentral**

In [ ]:
## FDC

# Create processor instance
bow_fdc_processor = TextSimilarityProcessor(df1=df_fooddatacentral_names, df2=df_foodon_names, vectorizer_type='bow')

# Compute shared dictionary
bow_fdc_processor.fit_vectorizer()

# Create BoW vectors specific for each dataset
bow_fdc_processor.transform_to_vec()

# Compute similarity between Nevo food items and FoodOn classes in matrix
bow_fdc_processor.compute_cosine_similarity()

fdc_names = list(set(fdc_labels['orig_name']))
bow_fdc_processor.find_top_k(names=fdc_names)

# Aggregate results
bow_fdc_processor.aggregate_results()
df_fdc_bow = bow_fdc_processor.get_results()
df_fdc_bow.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
df_fdc_bow.head()

In [ ]:
# FDC label matches
fdc_bow_merged_data = add_best_match_traditional(df_fdc_bow, fdc_labels, df_foodon_combined, 'orig_name')
fdc_bow_merged_data.head()

In [ ]:
bow_fdc_results = determine_results_traditional(fdc_bow_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
bow_fdc_results

**KAP**

In [ ]:
# Create processor instance
bow_kap_processor = TextSimilarityProcessor(df_kap_names, df_foodon_names, vectorizer_type='bow')

# Compute shared dictionary
bow_kap_processor.fit_vectorizer()

# Create BoW vectors specific for each dataset
bow_kap_processor.transform_to_vec()

# Compute similarity between Nevo food items and FoodOn classes in matrix
bow_kap_processor.compute_cosine_similarity()

kap_names = list(set(kap_labels['orig_name']))
bow_kap_processor.find_top_k(names=kap_names)

# Aggregate results
bow_kap_processor.aggregate_results()
df_kap_bow = bow_kap_processor.get_results()
df_kap_bow.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
df_kap_bow.head()

In [ ]:
# Nevo label matches
kap_bow_merged_data = add_best_match_traditional(df_kap_bow, kap_labels, df_foodon_combined, 'orig_name')
kap_bow_merged_data.head()

In [ ]:
bow_kap_results = determine_results_traditional(kap_bow_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
bow_kap_results

### Method 3: tf-idf

**NEVO**

In [ ]:
# Create processor instance
tfidf_nevo_processor = TextSimilarityProcessor(df_nevo_names, df_foodon_names, vectorizer_type='tfidf')

# Compute shared dictionary
tfidf_nevo_processor.fit_vectorizer()

# Create BoW vectors specific for each dataset
tfidf_nevo_processor.transform_to_vec()

# Compute similarity between Nevo food items and FoodOn classes in matrix
tfidf_nevo_processor.compute_cosine_similarity()

tfidf_nevo_processor.find_top_k(names=nevo_names)

# Aggregate results
tfidf_nevo_processor.aggregate_results()
df_nevo_tfidf = tfidf_nevo_processor.get_results()

df_nevo_tfidf.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
df_nevo_tfidf.head()

In [ ]:
# Nevo label matches
nevo_tfidf_merged_data = add_best_match_traditional(df_nevo_tfidf, nevo_labels, df_foodon_combined, 'nevo_name')
nevo_tfidf_merged_data.head()

In [ ]:
tfidf_nevo_results = determine_results_traditional(nevo_tfidf_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
tfidf_nevo_results

**Fooddatacentral**

In [ ]:
# Create processor instance
tfidf_fdc_processor = TextSimilarityProcessor(df_fooddatacentral_names, df_foodon_names, vectorizer_type='tfidf')

# Compute shared dictionary
tfidf_fdc_processor.fit_vectorizer()

# Create BoW vectors specific for each dataset
tfidf_fdc_processor.transform_to_vec()

# Compute similarity between fdc food items and FoodOn classes in matrix
tfidf_fdc_processor.compute_cosine_similarity()

tfidf_fdc_processor.find_top_k(names=fdc_names)

# Aggregate results
tfidf_fdc_processor.aggregate_results()
df_fdc_tfidf = tfidf_fdc_processor.get_results()

df_fdc_tfidf.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
df_fdc_tfidf.head()

In [ ]:
# FDC label matches
fdc_tfidf_merged_data = add_best_match_traditional(df_fdc_tfidf, fdc_labels, df_foodon_combined, 'orig_name')
fdc_tfidf_merged_data.head()

In [ ]:
tfidf_fdc_results = determine_results_traditional(fdc_tfidf_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
tfidf_fdc_results

**KAP**

In [ ]:
# Create processor instance
tfidf_kap_processor = TextSimilarityProcessor(df_kap_names, df_foodon_names, vectorizer_type='tfidf')

# Compute shared dictionary
tfidf_kap_processor.fit_vectorizer()

# Create BoW vectors specific for each dataset
tfidf_kap_processor.transform_to_vec()

# Compute similarity between Nevo food items and FoodOn classes in matrix
tfidf_kap_processor.compute_cosine_similarity()

tfidf_kap_processor.find_top_k(names=kap_names)

# Aggregate results
tfidf_kap_processor.aggregate_results()
df_kap_tfidf = tfidf_kap_processor.get_results()

df_kap_tfidf.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
df_kap_tfidf.head()

In [ ]:
# KAP label matches
kap_tfidf_merged_data = add_best_match_traditional(df_kap_tfidf, kap_labels, df_foodon_combined, 'orig_name')
kap_tfidf_merged_data.head()

In [ ]:
tfidf_kap_results = determine_results_traditional(kap_tfidf_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
tfidf_kap_results

## Method 4: SentenceBERT

In [ ]:
from helpers.step2 import BERTTextSimilarityProcessor

In [ ]:
# Set a random seed
random_seed = 42
random.seed(random_seed)

# Set a random seed for PyTorch (for GPU as well)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(random_seed)

model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

**Create embeddings**

In [ ]:
# FoodOn
sbert_embeddings_foodon = model_sbert.encode(df_foodon_names, batch_size=32, show_progress_bar=True)
df_foodon['bert_embeddings'] = sbert_embeddings_foodon.tolist()

# NEVO
sbert_embeddings_nevo = model_sbert.encode(df_nevo_names, batch_size=32, show_progress_bar=True)
df_nevo['bert_embeddings'] = sbert_embeddings_nevo.tolist()

# FoodData Central
sbert_embeddings_fdc = model_sbert.encode(df_fooddatacentral_names, batch_size=32, show_progress_bar=True)
df_fooddatacentral['bert_embeddings'] = sbert_embeddings_fdc.tolist()

# KAP
sbert_embeddings_kap = model_sbert.encode(df_kap_names, batch_size=32, show_progress_bar=True)
df_kap['bert_embeddings'] = sbert_embeddings_kap.tolist()

**NEVO**

In [ ]:
## NEVO

# Create processor instance
bert_nevo_processor = BERTTextSimilarityProcessor(df1=df_nevo, df2=df_foodon, df1_embedding_column='bert_embeddings', df2_embedding_column='bert_embeddings')

# Compute similarity between nevo food items and FoodOn classes in matrix 
bert_nevo_processor.compute_cosine_similarity()

# Find top 15 FoodOn matches
bert_nevo_processor.find_top_k(names=nevo_names)

# Aggregate results
bert_nevo_processor.aggregate_results()

# Get results as a DataFrame
df_bert_nevo_results = bert_nevo_processor.get_results()
df_bert_nevo_results.head()

In [ ]:
# Nevo label matches
df_bert_nevo_results.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
nevo_bert_merged_data = add_best_match_traditional(df_bert_nevo_results, nevo_labels, df_foodon_combined, 'nevo_name')
nevo_bert_merged_data.head()

In [ ]:
bert_nevo_results = determine_results_traditional(nevo_bert_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
bert_nevo_results

**FoodData Central**

In [ ]:
# Create processor instance
bert_fdc_processor = BERTTextSimilarityProcessor(df1=df_fooddatacentral, df2=df_foodon, df1_embedding_column='bert_embeddings', df2_embedding_column='bert_embeddings')

# Compute similarity between fdc food items and FoodOn classes in matrix 
bert_fdc_processor.compute_cosine_similarity()

# Find top 15 FoodOn matches
bert_fdc_processor.find_top_k(names=fdc_names)

# Aggregate results
bert_fdc_processor.aggregate_results()

# Get results as a DataFrame
df_bert_fdc_results = bert_fdc_processor.get_results()
df_bert_fdc_results.head(10)

In [ ]:
# Fdc label matches
df_bert_fdc_results.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
fdc_bert_merged_data = add_best_match_traditional(df_bert_fdc_results, fdc_labels, df_foodon_combined, 'orig_name')
fdc_bert_merged_data.head()

In [ ]:
bert_fdc_results = determine_results_traditional(fdc_bert_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
bert_fdc_results

**KAP**

In [ ]:
# Create processor instance
bert_kap_processor = BERTTextSimilarityProcessor(df1=df_kap, df2=df_foodon, df1_embedding_column='bert_embeddings', df2_embedding_column='bert_embeddings')

# Compute similarity between kap food items and FoodOn classes in matrix 
bert_kap_processor.compute_cosine_similarity()

# Find top 15 FoodOn matches
bert_kap_processor.find_top_k(names=kap_names)

# Aggregate results
bert_kap_processor.aggregate_results()

# Get results as a DataFrame
df_bert_kap_results = bert_kap_processor.get_results()
df_bert_kap_results.head(10)

In [ ]:
# KAP label matches
df_bert_kap_results.rename(columns={"df1_label": "orig_name", "df2_label": "candidates"}, inplace=True)
kap_bert_merged_data = add_best_match_traditional(df_bert_kap_results, kap_labels, df_foodon_combined, 'orig_name')
kap_bert_merged_data.head()

In [ ]:
bert_kap_results = determine_results_traditional(kap_bert_merged_data, labels = ['broader', 'exact', 'close match', 'related', 'narrow', 'unknown'])
bert_kap_results

### Save everything

In [ ]:
import pickle

# OpenAI results
with open("../data/intermediate/step2/nevo_openai_merged_data.pkl", 'wb') as f:
    pickle.dump(nevo_openai_merged_data, f)
with open("../data/intermediate/step2/fdc_openai_merged_data.pkl", 'wb') as f:
    pickle.dump(fdc_openai_merged_data, f)
with open("../data/intermediate/step2/kap_openai_merged_data.pkl", 'wb') as f:
    pickle.dump(kap_openai_merged_data, f)

# BoW results
with open("../data/intermediate/step2/nevo_bow_merged_data.pkl", 'wb') as f:
    pickle.dump(nevo_bow_merged_data, f)
with open("../data/intermediate/step2/fdc_bow_merged_data.pkl", 'wb') as f:
    pickle.dump(fdc_bow_merged_data, f)
with open("../data/intermediate/step2/kap_bow_merged_data.pkl", 'wb') as f:
    pickle.dump(kap_bow_merged_data, f)

# TF-IDF results
with open("../data/intermediate/step2/nevo_tfidf_merged_data.pkl", 'wb') as f:
    pickle.dump(nevo_tfidf_merged_data, f)
with open("../data/intermediate/step2/fdc_tfidf_merged_data.pkl", 'wb') as f:
    pickle.dump(fdc_tfidf_merged_data, f)
with open("../data/intermediate/step2/kap_tfidf_merged_data.pkl", 'wb') as f:
    pickle.dump(kap_tfidf_merged_data, f)

# BERT results
with open("../data/intermediate/step2/nevo_bert_merged_data.pkl", 'wb') as f:
    pickle.dump(nevo_bert_merged_data, f)
with open("../data/intermediate/step2/fdc_bert_merged_data.pkl", 'wb') as f:
    pickle.dump(fdc_bert_merged_data, f)
with open("../data/intermediate/step2/kap_bert_merged_data.pkl", 'wb') as f:
    pickle.dump(kap_bert_merged_data, f)

In [ ]:
# Save processor objects

# OpenAI processors
with open("../data/intermediate/step2/openai_nevo_processor.pkl", 'wb') as f:
    pickle.dump(openai_nevo_processor, f)
with open("../data/intermediate/step2/openai_fdc_processor.pkl", 'wb') as f:
    pickle.dump(openai_fdc_processor, f)
with open("../data/intermediate/step2/openai_kap_processor.pkl", 'wb') as f:
    pickle.dump(openai_kap_processor, f)

# BoW processors
with open("../data/intermediate/step2/bow_nevo_processor.pkl", 'wb') as f:
    pickle.dump(bow_nevo_processor, f)
with open("../data/intermediate/step2/bow_fdc_processor.pkl", 'wb') as f:
    pickle.dump(bow_fdc_processor, f)
with open("../data/intermediate/step2/bow_kap_processor.pkl", 'wb') as f:
    pickle.dump(bow_kap_processor, f)

# TF-IDF processors
with open("../data/intermediate/step2/tfidf_nevo_processor.pkl", 'wb') as f:
    pickle.dump(tfidf_nevo_processor, f)
with open("../data/intermediate/step2/tfidf_fdc_processor.pkl", 'wb') as f:
    pickle.dump(tfidf_fdc_processor, f)
with open("../data/intermediate/step2/tfidf_kap_processor.pkl", 'wb') as f:
    pickle.dump(tfidf_kap_processor, f)

# BERT processors
with open("../data/intermediate/step2/bert_nevo_processor.pkl", 'wb') as f:
    pickle.dump(bert_nevo_processor, f)
with open("../data/intermediate/step2/bert_fdc_processor.pkl", 'wb') as f:
    pickle.dump(bert_fdc_processor, f)
with open("../data/intermediate/step2/bert_kap_processor.pkl", 'wb') as f:
    pickle.dump(bert_kap_processor, f)